In [11]:
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
processed_dir = "../data/processed"

In [5]:
X_train = pd.read_csv(os.path.join(processed_dir, "X_train_new.csv"))
X_test = pd.read_csv(os.path.join(processed_dir, "X_test_new.csv"))
y_train = pd.read_csv(os.path.join(processed_dir, "y_train_new.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(processed_dir, "y_test_new.csv")).squeeze("columns")

In [6]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (534932, 19)
X_test shape:  (133733, 19)
y_train shape: (534932,)
y_test shape:  (133733,)


In [7]:
X = pd.concat([X_train], axis=0).reset_index(drop=True)
y = pd.concat([y_train], axis=0).reset_index(drop=True)

In [8]:
neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight_val = neg_count / pos_count
print(f"Scale Pos Weight value: {scale_pos_weight_val:.2f}")

Scale Pos Weight value: 4.73


In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
fold_aucs = []
fold_accuracies = []

all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=scale_pos_weight_val,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Out-of-Fold) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))


--- Training Fold 1 ---
Fold 1 Accuracy: 0.8513 | ROC-AUC: 0.9410

--- Training Fold 2 ---
Fold 2 Accuracy: 0.8549 | ROC-AUC: 0.9414

--- Training Fold 3 ---
Fold 3 Accuracy: 0.8507 | ROC-AUC: 0.9407

--- Training Fold 4 ---
Fold 4 Accuracy: 0.8531 | ROC-AUC: 0.9408

--- Training Fold 5 ---
Fold 5 Accuracy: 0.8523 | ROC-AUC: 0.9415

Mean Accuracy across 5 Folds: 0.8524
Mean ROC-AUC across 5 Folds:  0.9411

--- Overall Classification Report (Out-of-Fold) ---
              precision    recall  f1-score   support

           0     0.9788    0.8394    0.9038    441509
           1     0.5464    0.9141    0.6839     93423

    accuracy                         0.8524    534932
   macro avg     0.7626    0.8767    0.7938    534932
weighted avg     0.9033    0.8524    0.8654    534932



In [16]:
from sklearn.svm import LinearSVC

In [43]:
fold_accuracies = []
fold_aucs = []

all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Linear SVM Fold {fold + 1} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    model = LinearSVC(C=50, class_weight='balanced', max_iter=2000, random_state=42)
    
    model.fit(X_tr_scaled, y_tr)
    
    preds = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, preds)
    fold_accuracies.append(acc)
    
    scores = model.decision_function(X_val_scaled)
    auc = roc_auc_score(y_val, scores)
    fold_aucs.append(auc)
    
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Out-of-Fold) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))


--- Training Linear SVM Fold 1 ---
Fold 1 Accuracy: 0.8549 | ROC-AUC: 0.9379

--- Training Linear SVM Fold 2 ---
Fold 2 Accuracy: 0.8554 | ROC-AUC: 0.9382

--- Training Linear SVM Fold 3 ---
Fold 3 Accuracy: 0.8532 | ROC-AUC: 0.9376

--- Training Linear SVM Fold 4 ---
Fold 4 Accuracy: 0.8538 | ROC-AUC: 0.9378

--- Training Linear SVM Fold 5 ---
Fold 5 Accuracy: 0.8545 | ROC-AUC: 0.9385

Mean Accuracy across 5 Folds: 0.8544
Mean ROC-AUC across 5 Folds:  0.9380

--- Overall Classification Report (Out-of-Fold) ---
              precision    recall  f1-score   support

           0     0.9753    0.8450    0.9055    441509
           1     0.5509    0.8987    0.6831     93423

    accuracy                         0.8544    534932
   macro avg     0.7631    0.8718    0.7943    534932
weighted avg     0.9012    0.8544    0.8666    534932



In [24]:
from sklearn.tree import DecisionTreeClassifier

In [41]:
dt_fold_accuracies = []
dt_fold_aucs = []
feature_importances_list = []

all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Decision Tree Fold {fold + 1} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=4)
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    dt_fold_accuracies.append(acc)
    dt_fold_aucs.append(auc)
    feature_importances_list.append(model.feature_importances_)
    
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(dt_fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(dt_fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Out-of-Fold) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))

mean_importances = np.mean(feature_importances_list, axis=0)
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': mean_importances
}).sort_values(by='Importance', ascending=False)

print("\n--- أهم الميزات (Feature Importances) في نموذج Decision Tree ---")
print(feature_importance_df.to_string(index=False))


--- Training Decision Tree Fold 1 ---
Fold 1 Accuracy: 0.8490 | ROC-AUC: 0.9369

--- Training Decision Tree Fold 2 ---
Fold 2 Accuracy: 0.8476 | ROC-AUC: 0.9377

--- Training Decision Tree Fold 3 ---
Fold 3 Accuracy: 0.8476 | ROC-AUC: 0.9363

--- Training Decision Tree Fold 4 ---
Fold 4 Accuracy: 0.8461 | ROC-AUC: 0.9373

--- Training Decision Tree Fold 5 ---
Fold 5 Accuracy: 0.8460 | ROC-AUC: 0.9379

Mean Accuracy across 5 Folds: 0.8472
Mean ROC-AUC across 5 Folds:  0.9372

--- Overall Classification Report (Out-of-Fold) ---
              precision    recall  f1-score   support

           0     0.9796    0.8322    0.8999    441509
           1     0.5366    0.9183    0.6774     93423

    accuracy                         0.8472    534932
   macro avg     0.7581    0.8752    0.7887    534932
weighted avg     0.9023    0.8472    0.8611    534932


--- أهم الميزات (Feature Importances) في نموذج Decision Tree ---
                    Feature   Importance
Environmental_Concern_Level 5.772

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [42]:
lr_fold_accuracies = []
lr_fold_aucs = []

all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Logistic Regression Fold {fold + 1} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_tr_scaled, y_tr)
    
    preds = model.predict(X_val_scaled)
    probs = model.predict_proba(X_val_scaled)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    lr_fold_accuracies.append(acc)
    lr_fold_aucs.append(auc)
    
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(lr_fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(lr_fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Out-of-Fold) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))


--- Training Logistic Regression Fold 1 ---
Fold 1 Accuracy: 0.8576 | ROC-AUC: 0.9380

--- Training Logistic Regression Fold 2 ---
Fold 2 Accuracy: 0.8581 | ROC-AUC: 0.9384

--- Training Logistic Regression Fold 3 ---
Fold 3 Accuracy: 0.8561 | ROC-AUC: 0.9377

--- Training Logistic Regression Fold 4 ---
Fold 4 Accuracy: 0.8569 | ROC-AUC: 0.9379

--- Training Logistic Regression Fold 5 ---
Fold 5 Accuracy: 0.8572 | ROC-AUC: 0.9386

Mean Accuracy across 5 Folds: 0.8572
Mean ROC-AUC across 5 Folds:  0.9381

--- Overall Classification Report (Out-of-Fold) ---
              precision    recall  f1-score   support

           0     0.9745    0.8492    0.9076    441509
           1     0.5567    0.8949    0.6864     93423

    accuracy                         0.8572    534932
   macro avg     0.7656    0.8721    0.7970    534932
weighted avg     0.9015    0.8572    0.8689    534932



In [38]:
y.value_counts()

Will_Buy_EV
0    441509
1     93423
Name: count, dtype: int64

In [47]:
from imblearn.under_sampling import RandomUnderSampler

In [48]:
fold_accuracies = []
fold_aucs = []
all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training XGBoost with Under-Sampling Fold {fold + 1} ---")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
    X_tr_resampled, y_tr_resampled = rus.fit_resample(X_tr, y_tr)
    
    print(f"عدد العينات قبل التوازن: {X_tr.shape[0]} | بعد الـ Under-sampling: {X_tr_resampled.shape[0]}")
    
    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    
    model.fit(X_tr_resampled, y_tr_resampled)
    
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Out-of-Fold) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))


--- Training XGBoost with Under-Sampling Fold 1 ---
عدد العينات قبل التوازن: 427945 | بعد الـ Under-sampling: 149476
Fold 1 Accuracy: 0.8526 | ROC-AUC: 0.9398

--- Training XGBoost with Under-Sampling Fold 2 ---
عدد العينات قبل التوازن: 427945 | بعد الـ Under-sampling: 149476
Fold 2 Accuracy: 0.8542 | ROC-AUC: 0.9397

--- Training XGBoost with Under-Sampling Fold 3 ---
عدد العينات قبل التوازن: 427946 | بعد الـ Under-sampling: 149478
Fold 3 Accuracy: 0.8515 | ROC-AUC: 0.9390

--- Training XGBoost with Under-Sampling Fold 4 ---
عدد العينات قبل التوازن: 427946 | بعد الـ Under-sampling: 149478
Fold 4 Accuracy: 0.8529 | ROC-AUC: 0.9393

--- Training XGBoost with Under-Sampling Fold 5 ---
عدد العينات قبل التوازن: 427946 | بعد الـ Under-sampling: 149476
Fold 5 Accuracy: 0.8545 | ROC-AUC: 0.9402

Mean Accuracy across 5 Folds: 0.8531
Mean ROC-AUC across 5 Folds:  0.9396

--- Overall Classification Report (Out-of-Fold) ---
              precision    recall  f1-score   support

           0     

In [49]:
threshold = 0.001
selected_features = feature_importance_df[feature_importance_df['Importance'] > threshold]['Feature'].tolist()

print(f"عدد الميزات الأصلية: {X.shape[1]}")
print(f"عدد الميزات بعد التصفية: {len(selected_features)}")
print(f"الميزات المختارة: {selected_features}")

X_selected = X[selected_features]

fold_accuracies = []
fold_aucs = []
all_y_true = []
all_y_pred = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_selected, y)):
    print(f"\n--- Training Decision Tree (Selected Features) Fold {fold + 1} ---")
    
    X_tr, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=4)
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    all_y_true.extend(y_val)
    all_y_pred.extend(preds)
    
    print(f"Fold {fold + 1} Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

print("\n================================")
print(f"Mean Accuracy across 5 Folds: {np.mean(fold_accuracies):.4f}")
print(f"Mean ROC-AUC across 5 Folds:  {np.mean(fold_aucs):.4f}")
print("================================")

print("\n--- Overall Classification Report (Selected Features) ---")
print(classification_report(all_y_true, all_y_pred, digits=4))

عدد الميزات الأصلية: 19
عدد الميزات بعد التصفية: 8
الميزات المختارة: ['Environmental_Concern_Level', 'Subsidy_Available', 'Annual_Income_USD', 'Range_Anxiety_Level', 'Commute_Anxiety_Interaction', 'Age', 'Daily_Commute_km', 'Charging_Stations_Near_Work']

--- Training Decision Tree (Selected Features) Fold 1 ---
Fold 1 Accuracy: 0.8465 | ROC-AUC: 0.9371

--- Training Decision Tree (Selected Features) Fold 2 ---
Fold 2 Accuracy: 0.8479 | ROC-AUC: 0.9372

--- Training Decision Tree (Selected Features) Fold 3 ---
Fold 3 Accuracy: 0.8475 | ROC-AUC: 0.9370

--- Training Decision Tree (Selected Features) Fold 4 ---
Fold 4 Accuracy: 0.8464 | ROC-AUC: 0.9370

--- Training Decision Tree (Selected Features) Fold 5 ---
Fold 5 Accuracy: 0.8468 | ROC-AUC: 0.9379

Mean Accuracy across 5 Folds: 0.8470
Mean ROC-AUC across 5 Folds:  0.9372

--- Overall Classification Report (Selected Features) ---
              precision    recall  f1-score   support

           0     0.9797    0.8318    0.8998    4415